# Project Planner Quickstart

This notebook demonstrates the core features of project_planner, including scheduling, cost tracking, resource loading, and leveling.

## Step 1: Import Required Modules

Start by importing the main module and required standard library types.

In [ ]:
import project_planner as pp
from datetime import date
from decimal import Decimal

## Step 2: Create a Workspace and Project

Open an in-memory workspace (good for notebooks), create a new project named "Demo", and initialize the calendar with working hours and a holiday.

In [ ]:
ws = pp.open_workspace(":memory:")
ws.new_project("Demo", start=date(2026, 10, 5))
ws.calendar.initialize(hours_per_day=8, working_days_per_year=220, holidays=[(date(2026, 10, 8), "Holiday")])
ws.calendar

## Step 3: Add Resources and Create Tasks

Add two resources (Alice at $100/h and Bob at $50/h), create a project group "Phase 1", and define tasks with effort and duration.

In [ ]:
alice = ws.add_resource("Alice", hourly_rate="100")
bob = ws.add_resource("Bob", hourly_rate="50")
phase1 = ws.add_group("Phase 1")

design = ws.add_task("Design", parent=phase1, effort=pp.hours(40))
ws.set_assignment(design, alice, percent=80)
ws.set_assignment(design, bob, percent=20)

build = ws.add_task("Build", parent=phase1, duration=pp.days(2))

## Step 4: Add Dependencies and Schedule

Create a Finish-to-Start dependency from Design to Build with a -0.5 day lag, then schedule the project.

In [ ]:
ws.add_dependency(design, build, "FS", lag=pp.days(-0.5))

result = ws.schedule()
result

### Schedule Summary

Check the schedule completeness, project finish date, and total cost.

In [ ]:
print(f"Complete: {result.complete}")
print(f"Project finish: {result.project_finish}")
print(f"Total cost: {result.total_cost}")

## Step 5: Cost Analysis

Generate cost reports in different units and inspect task details.

In [ ]:
ws.cost_report(unit="person_days")

In [ ]:
ws.cost_report(unit="person_hours")

In [ ]:
ws.task_details(design)

## Step 6: Resource Loading and Overload Scenario

Check Alice's loading in the current schedule, then add an overlapping Review task assigned 60% to Alice, creating an overload.

In [ ]:
ws.loading(alice)

Add a Review task that overlaps with Design, creating a resource overload.

In [ ]:
review = ws.add_task("Review", parent=phase1, duration=pp.days(3))
ws.set_assignment(review, alice, percent=60)

# recalculate so the new task is in the schedule, then inspect Alice's loading
before = ws.schedule()
print(f"Cost before leveling: {before.total_cost}")
loading = ws.loading(alice)
print(f"Peak load: {max(s.percent for s in loading.segments)}%")
loading

## Step 7: Leveling Preview and Application

Preview the leveling solution to resolve the overload, inspect the delays, and then apply the leveled schedule.

In [ ]:
preview = ws.level_preview()
preview.delays_days

In [ ]:
ws.apply_leveling()
leveled_result = ws.result()
print(f"New project finish: {leveled_result.project_finish}")
print(f"Cost after leveling: {leveled_result.total_cost}")

## Step 8: Save and List Projects

Save the project and list all saved projects in the workspace.

In [ ]:
ws.save_as("Demo")
ws.list_projects()

## Step 9: CSV Export and Import

Export the project to CSV, create a new workspace, import the CSV, and verify the schedule is identical.

In [ ]:
text = ws.export_csv()
print("First 500 characters of CSV export:")
print(text[:500])

In [ ]:
ws2 = pp.open_workspace(":memory:")
ws2.import_csv(text)
result2 = ws2.schedule()
print(f"Imported project cost: {result2.total_cost}")